# Sobrino1993 — Split-window

Aplica uma formulação histórica de AVHRR a entradas Landsat como adaptação experimental. Os coeficientes não constituem uma calibração específica de TIRS.

## Precisa de

**Temperaturas de brilho — obrigatórias:**

- **BT10:** imagem de temperatura de brilho TOA da banda 10, em **K (kelvin)**.
- **BT11:** imagem de temperatura de brilho TOA da banda 11, em **K (kelvin)**.

Não fornecer DN, radiância, LST ou temperaturas em Celsius nesses dois parâmetros.

**Emissividade — escolha uma das duas formas:**

**1. Calcular dentro da função, como neste notebook:**

- **Vermelho:** imagem calibrada em radiância TOA, em **W m⁻² sr⁻¹ µm⁻¹**.
- **Infravermelho próximo (NIR):** imagem calibrada na **mesma grandeza e unidade do vermelho**.

O NDVI é calculado internamente; não precisa ser fornecido separadamente. A seleção automática usa `emissivity_method="gopinadh-2018"`. A convenção radiométrica do exemplo deve ser compatível com a parametrização.

**2. Fornecer emissividades já calculadas:**

- **`emissivity_10`:** imagem de emissividade da banda 10, **adimensional**, com valores em **(0,1]**.
- **`emissivity_11`:** imagem de emissividade da banda 11, **adimensional**, com valores em **(0,1]**.

Fornecer sempre as duas emissividades, específicas das respectivas bandas, e omitir `emissivity_method`.

Com emissividades prontas, omitir também vermelho/NIR. As duas formas de entrada não devem ser combinadas.

**Entrada opcional de qualidade:**

- **`mask`:** imagem **booleana**, sem unidade: `True` exclui o pixel; `False` permite avaliar, sujeito às validações do método. Neste exemplo, sua origem é descrita na seção de leitura.

Todas as imagens usadas juntas devem ter a mesma quantidade de linhas/colunas, posição dos pixels, resolução e sistema de coordenadas.

## Arquivos para executar este notebook

O exemplo usa o leitor de cena **Landsat 8/9 Collection 2 Level 1**. Coloque na pasta da cena:

- **B4:** arquivo original do vermelho, em **DN**; o leitor converte para radiância TOA, em W m⁻² sr⁻¹ µm⁻¹.
- **B5:** arquivo original do NIR, em **DN**; o leitor converte para radiância TOA, em W m⁻² sr⁻¹ µm⁻¹.
- **B10:** arquivo térmico original, em **DN**; o leitor converte para BT10 em **K**.
- **B11:** arquivo térmico original, em **DN**; o leitor converte para BT11 em **K**.
- **MTL.txt:** metadados e coeficientes de calibração da própria cena; não é uma imagem.
- **QA_PIXEL:** códigos inteiros de qualidade, sem unidade física; o leitor interpreta os bits para montar a máscara.
- **QA_RADSAT:** códigos inteiros de saturação e oclusão de terreno, sem unidade física; complementam a máscara.

Essa lista corresponde ao leitor usado no notebook; uma chamada direta à função exige somente as entradas indicadas em “Precisa de”. B2 e SZA são opcionais para o leitor. Preserve os nomes originais e a grade dos arquivos. Não passar os arquivos QA brutos como máscara booleana.

## Como as imagens são preparadas

**TOA significa topo da atmosfera.** A grandeza também precisa ser especificada:

- **Radiância TOA:** energia medida pelo sensor, em **W m⁻² sr⁻¹ µm⁻¹**.
- **Reflectância TOA:** **adimensional**, com ajuste do ângulo solar.
- **Temperatura de brilho TOA:** temperatura equivalente da radiância térmica, em **K**.

O leitor usa o **MTL da própria cena** para calibrar os arquivos originais. Não é necessário preparar manualmente as imagens TOA para executar este notebook.

A configuração `optical_quantity="radiance"` devolve as bandas ópticas em **radiância TOA, em W m⁻² sr⁻¹ µm⁻¹**; as bandas térmicas são devolvidas como temperaturas de brilho em **K**.

Trocar para `optical_quantity="reflectance"` muda a calibração óptica; isso exige verificar a compatibilidade com a parametrização usada no método. **Reflectância TOA ainda contém efeitos atmosféricos e é diferente de reflectância de superfície.**

Edite os caminhos na célula de configuração e execute de cima para baixo. As saídas ficam em `resultados/`; os arquivos originais não são modificados.


## Função geral — parâmetros aplicáveis ao método

Argumentos nomeados e relacionados ficam próximos. A tabela abaixo segue **exatamente a ordem desta chamada**. Obrigatório: necessário ao cálculo; opcional: pode usar o padrão; condicional: depende da rota escolhida. Valores de configuração mostrados podem diferir do padrão da API.

```python
resultado = split_window(
    # Método
    lst_method='sobrino-1993',

    # Emissividade e suas entradas
    emissivity_method='gopinadh-2018',
    band_4_red=opticas[4],
    band_5_nir=opticas[5],
    optical_quantity='radiance',

    # Temperaturas de brilho — K
    brightness_band_10=bt[10],
    brightness_band_11=bt[11],

    # Exclusão de pixels
    mask=mascara,

    # Saída
    unit='celsius',
    return_diagnostics=False,
)
```

### Detalhamento dos parâmetros

| Parâmetro | Condição | Padrão da API | Unidade | Detalhamento e opções |
|---|---|---|---|---|
| `lst_method` | Obrigatório neste exemplo: selecionar o método | `None (precisa informar)` | Seletor | Usar "sobrino-1993" neste notebook. Os outros métodos têm requisitos próprios e notebooks separados. |
| `emissivity_method` | Opcional na rota por bandas | `None → "gopinadh-2018"` | Seletor | Usar "gopinadh-2018" no cálculo automático de split-window; "avdan-2016" é bloqueado. Para outro modelo compatível, calcular emissividades por banda separadamente e fornecer o par pronto. |
| `band_4_red` | Obrigatório nesta rota; omitir com emissividades prontas | `None` | W m⁻² sr⁻¹ µm⁻¹ neste exemplo | Banda vermelha calibrada e alinhada. Mesma grandeza radiométrica da outra banda; não misturar DN, radiância e reflectância. NDVI calculado internamente. Respeitar a convenção óptica indicada na introdução. |
| `band_5_nir` | Obrigatório nesta rota; omitir com emissividades prontas | `None` | W m⁻² sr⁻¹ µm⁻¹ neste exemplo | Banda NIR calibrada e alinhada. Mesma grandeza radiométrica da outra banda; não misturar DN, radiância e reflectância. NDVI calculado internamente. Respeitar a convenção óptica indicada na introdução. |
| `optical_quantity` | Opcional; recomendado declarar | `None` | Seletor | "radiance", "reflectance"/"toa_reflectance" ou "surface_reflectance". Declara a grandeza; não converte as bandas nem faz correção atmosférica. |
| `brightness_band_10` | Obrigatório | `Sem padrão` | K | Temperatura de brilho TOA B10, calibrada pelo MTL. Não passar DN, radiância, LST ou Celsius. Mesma forma e grade das demais imagens. |
| `brightness_band_11` | Obrigatório | `Sem padrão` | K | Temperatura de brilho TOA B11, calibrada pelo MTL. Não passar DN, radiância, LST ou Celsius. Mesma forma e grade das demais imagens. |
| `mask` | Opcional; recomendado | `None` | Booleano | Máscara de exclusão de pixels, alinhada às imagens: True rejeita o pixel (NaN); False permite a avaliação, sujeita às validações do método. Não fornecer a banda QA bruta. A origem da máscara deste exemplo está descrita na seção de leitura. |
| `unit` | Opcional | `"kelvin"` | K ou °C | "kelvin" ou "celsius"; altera somente a saída. Temperaturas de brilho entram sempre em K. |
| `return_diagnostics` | Opcional | `False` | Booleano | False retorna somente a imagem; True retorna dicionário com: lst, ndvi, emissivity_10, emissivity_11, invalid_mask, units, optical_contract. A máscara final pode incluir rejeições adicionais à máscara de entrada. |

## Alternativa — emissividades prontas

Calcule e10/e11 separadamente, com parâmetros verificados de cada banda e mesma grade das temperaturas. Não fornecer emissivity_method nem bandas ópticas nesta chamada.

```python
resultado = split_window(
    # Método
    lst_method='sobrino-1993',

    # Emissividade e suas entradas
    emissivity_10=e10,
    emissivity_11=e11,

    # Temperaturas de brilho — K
    brightness_band_10=bt[10],
    brightness_band_11=bt[11],

    # Exclusão de pixels
    mask=mascara,

    # Saída
    unit='celsius',
    return_diagnostics=False,
)
```

### Detalhamento dos parâmetros

| Parâmetro | Condição | Padrão da API | Unidade | Detalhamento e opções |
|---|---|---|---|---|
| `lst_method` | Obrigatório neste exemplo: selecionar o método | `None (precisa informar)` | Seletor | Usar "sobrino-1993" neste notebook. Os outros métodos têm requisitos próprios e notebooks separados. |
| `emissivity_10` | Condicional: obrigatório com o par pronto | `None` | Adimensional, (0,1] | Imagem de emissividade B10 alinhada a BT10/BT11, específica da banda. Fornecer as duas emissividades; omitir emissivity_method e as bandas ópticas. Kerr ainda exige NDVI preparado ou bandas ópticas para seu fator de vegetação. |
| `emissivity_11` | Condicional: obrigatório com o par pronto | `None` | Adimensional, (0,1] | Imagem de emissividade B11 alinhada a BT10/BT11, específica da banda. Fornecer as duas emissividades; omitir emissivity_method e as bandas ópticas. Kerr ainda exige NDVI preparado ou bandas ópticas para seu fator de vegetação. |
| `brightness_band_10` | Obrigatório | `Sem padrão` | K | Temperatura de brilho TOA B10, calibrada pelo MTL. Não passar DN, radiância, LST ou Celsius. Mesma forma e grade das demais imagens. |
| `brightness_band_11` | Obrigatório | `Sem padrão` | K | Temperatura de brilho TOA B11, calibrada pelo MTL. Não passar DN, radiância, LST ou Celsius. Mesma forma e grade das demais imagens. |
| `mask` | Opcional; recomendado | `None` | Booleano | Máscara de exclusão de pixels, alinhada às imagens: True rejeita o pixel (NaN); False permite a avaliação, sujeita às validações do método. Não fornecer a banda QA bruta. A origem da máscara deste exemplo está descrita na seção de leitura. |
| `unit` | Opcional | `"kelvin"` | K ou °C | "kelvin" ou "celsius"; altera somente a saída. Temperaturas de brilho entram sempre em K. |
| `return_diagnostics` | Opcional | `False` | Booleano | False retorna somente a imagem; True retorna dicionário com: lst, ndvi, emissivity_10, emissivity_11, invalid_mask, units, optical_contract. A máscara final pode incluir rejeições adicionais à máscara de entrada. |


In [ ]:
from pathlib import Path
import numpy as np
from pylstemp import split_window
from pylstemp.scene import LandsatScene

pasta = Path("CAMINHO_DA_CENA")
saida = Path("resultados/sobrino_1993")
saida.mkdir(parents=True, exist_ok=True)
unidade = "celsius"
diagnosticos = False

## Ler e calibrar a imagem

`LandsatScene` confere os arquivos/grade da cena e le o MTL. `calibrated()` retorna bandas opticas, BT10/BT11 em K e mascara QA booleana. As entradas deste exemplo sao radiancias opticas TOA.

### Máscara usada neste exemplo

`mask` é uma máscara **de exclusão**, não uma máscara de seleção: `True` rejeita o pixel e `False` permite o cálculo. O leitor usa QA_PIXEL para fill, nuvem dilatada, cirrus de alta confiança, nuvem, sombra e neve; QA_RADSAT para saturação óptica B2/B4/B5 e oclusão de terreno; e verifica SZA quando disponível. A opção `exclude_water=True` também exclui água (bit7 de QA_PIXEL). Valores não finitos ou fora do domínio podem ser rejeitados adicionalmente pelo cálculo. `mask=None` em uma chamada direta não cria automaticamente uma máscara de nuvens.


In [ ]:
cena = LandsatScene(pasta)
opticas, bt, mascara = cena.calibrated(optical_quantity="radiance")

## Aplicar o metodo

Nao e uma parametrizacao calibrada para TIRS. Use para comparacoes metodologicas com esta limitacao explicita.

In [ ]:
resultado = split_window(
    # Método
    lst_method='sobrino-1993',

    # Emissividade e suas entradas
    emissivity_method='gopinadh-2018',
    band_4_red=opticas[4],
    band_5_nir=opticas[5],
    optical_quantity='radiance',

    # Temperaturas de brilho — K
    brightness_band_10=bt[10],
    brightness_band_11=bt[11],

    # Exclusão de pixels
    mask=mascara,

    # Saída
    unit=unidade,
    return_diagnostics=diagnosticos,
)
imagem = resultado['lst'] if diagnosticos else resultado


## Salvar a imagem

A saida preserva a grade original e e gravada como float32 com NoData NaN. A unidade da emissividade/reflectancia e adimensional; PWV usa g/cm2; BT usa K e LST respeita `unit`.

In [ ]:
cena.write_geotiff(saida / "sobrino_1993.tif", imagem, units=unidade)
cena.close()
print("Pixels validos:", int(np.isfinite(imagem).sum()), "de", imagem.size)
print("Saida:", saida)

## Referencia e limites

Veja a [ficha metodologica](../docs/method-cards.md) para equacao, coeficientes, dominio e adaptacoes. Execucao numericamente correta nao estabelece acuracia de campo. [Voltar ao indice](README.md).